<a href="https://colab.research.google.com/github/MarceloGMW/humidity-detection-aeco/blob/main/MAIC1125_M4T3_TAREA_MarceloMelluso_BenoitCourbin_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ AECO CV — Detección de Manchas de Humedad con YOLOv8
**Autores:** Marcelo Melluso & Benoit Courbin  
**Módulo:** MAIC1125 — M4T3  
**Dataset:** [Detección de Humedades — Roboflow Universe](https://universe.roboflow.com/marcelos-workspace-rfie2/deteccion-de-humedades/dataset/1)  
**Modelo:** YOLOv8s (detección de objetos)  

---
## 📋 Cómo Reproducir (end-to-end)

1. Abre este notebook en Google Colab: `Archivo → Abrir cuaderno → GitHub` o sube directamente
2. Activa GPU: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`
3. Añade tu API Key de Roboflow en los **Secrets de Colab** (🔑 icono candado en barra lateral) con el nombre `ROBOFLOW_API_KEY`
4. `Entorno de ejecución → Ejecutar todo`

**Parámetros del experimento:**
- Dataset: `marcelos-workspace-rfie2 / deteccion-de-humedades / v2` (split 80/20)
- Modelo: `yolov8s` (small)
- Épocas: `50` (early stopping patience=10)
- Batch: `16` | imgsz: `640`
- Ultralytics: `8.x` (ver pip freeze al final)

**Tiempo estimado:** ~25-40 min con GPU T4  
**Última ejecución exitosa:** 2026-03-05 — GPU T4 Colab

---
# 0. ⚙️ Entorno y Dependencias

In [ ]:
# ── Instalaciones
!pip install ultralytics roboflow --quiet

import ultralytics
ultralytics.checks()
print(f"✅ Ultralytics {ultralytics.__version__}")

In [ ]:
# ── Imports
import os, json, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from IPython.display import display

from ultralytics import YOLO

# ── Verificar GPU
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Dispositivo: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  Sin GPU — el entrenamiento será lento. Activa T4 en Entorno de ejecución.")

---
# 1. 📦 Descarga del Dataset desde Roboflow

In [ ]:
# ── API Key desde Secrets de Colab (recomendado) o variable directa
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KE')
    print("✅ API Key cargada desde Secrets de Colab")
except Exception:
    # Fallback: poner la key directamente (NO subir a GitHub público)
    ROBOFLOW_API_KEY = serdata.get('ROBOFLOW_API_KE')  # <-- reemplazar si es necesario
    print("⚠️  API Key cargada directamente — recuerda no subir esto a GitHub público")

# ── Descarga del dataset en formato YOLOv8
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace("marcelos-workspace-rfie2").project("humedades-edilicias-iii")
dataset = project.version(3).download("yolov8")

DATASET_PATH = dataset.location
DATA_YAML    = os.path.join(DATASET_PATH, "data.yaml")

print(f"\n📁 Dataset descargado en : {DATASET_PATH}")
print(f"📄 data.yaml             : {DATA_YAML}")

In [ ]:
# ── Inspección del dataset
import yaml

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

CLASS_NAMES = data_cfg.get("names", [])
NUM_CLASSES = data_cfg.get("nc", len(CLASS_NAMES))

print("\n" + "="*50)
print("  CONFIGURACIÓN DEL DATASET")
print("="*50)
print(f"  Clases ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"  Split  : 80% train / 20% val")

# Contar imágenes por split
for split in ["train", "valid", "test"]:
    img_dir = Path(DATASET_PATH) / split / "images"
    if img_dir.exists():
        n = len(list(img_dir.glob("*.*")))
        print(f"  {split:6s}: {n} imágenes")
print("="*50)

In [ ]:
# ── Visualización de ejemplos de anotación (3-5 imágenes)
from ultralytics.utils.plotting import Annotator
import random, cv2

RESULTS_DIR = Path("/content/results/evidence")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train_img_dir   = Path(DATASET_PATH) / "train" / "images"
train_label_dir = Path(DATASET_PATH) / "train" / "labels"

sample_imgs = random.sample(list(train_img_dir.glob("*.jpg")) +
                             list(train_img_dir.glob("*.png")), k=min(5, len(list(train_img_dir.iterdir()))))

fig, axes = plt.subplots(1, len(sample_imgs), figsize=(5*len(sample_imgs), 5))
if len(sample_imgs) == 1:
    axes = [axes]

COLORS = plt.cm.tab10(np.linspace(0, 1, max(NUM_CLASSES, 1)))

for ax, img_path in zip(axes, sample_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    label_path = train_label_dir / (img_path.stem + ".txt")
    if label_path.exists():
        with open(label_path) as lf:
            for line in lf:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    color = tuple((np.array(COLORS[cls_id % len(COLORS)][:3]) * 255).astype(int).tolist())
                    cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                    label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
                    cv2.putText(img, label, (x1, max(y1-5,0)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")

plt.suptitle("Ejemplos de Anotación — Dataset de Entrenamiento", fontsize=13, fontweight="bold")
plt.tight_layout()
annotation_path = RESULTS_DIR / "annotation_examples.png"
plt.savefig(annotation_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Guardado: {annotation_path}")

---
# 2. 🚀 Entrenamiento YOLOv8

In [ ]:
# ── 2a. Corregir data.yaml para que Ultralytics encuentre las imágenes
import yaml

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("data.yaml original:")
print(json.dumps(data_cfg, indent=2, default=str))

# Ultralytics necesita rutas absolutas apuntando a las carpetas de imágenes
data_cfg["path"]  = DATASET_PATH                          # raíz del dataset
data_cfg["train"] = "train/images"
data_cfg["val"]   = "valid/images"      # Roboflow usa 'valid', no 'val'

# Añadir test si existe
test_dir = Path(DATASET_PATH) / "test" / "images"
if test_dir.exists():
    data_cfg["test"] = "test/images"

with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, allow_unicode=True)

print("\ndata.yaml corregido:")
with open(DATA_YAML) as f:
    print(f.read())

# Verificar que las rutas existen
for split in ["train", "valid", "test"]:
    p = Path(DATASET_PATH) / split / "images"
    status = "✅" if p.exists() else "⚠️  NO EXISTE"
    print(f"  {status}  {p}")

# ── Configuración del experimento
EXP_CFG = {
    "model"    : "yolov8s.pt",   # variante small — buen equilibrio velocidad/precisión
    "epochs"   :100,
    "batch"    : 16,
    "imgsz"    : 640,
    "patience" :50,             # early stopping
    "project"  : "/content/runs/detect",
    "name"     : "humedad_yolov8s",
    "device"   : DEVICE,
    "exist_ok" : True,
}

print("\n" + "="*50)
print("  CONFIGURACIÓN DEL ENTRENAMIENTO")
print("="*50)
for k, v in EXP_CFG.items():
    print(f"  {k:<12}: {v}")
print("="*50)

In [ ]:
# ── Entrenamiento
model = YOLO(EXP_CFG["model"])

print(f"\n🚀 Iniciando entrenamiento — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"   Modelo  : {EXP_CFG['model']}")
print(f"   Épocas  : {EXP_CFG['epochs']} (patience={EXP_CFG['patience']})")
print(f"   Dataset : {DATA_YAML}\n")

results_train = model.train(
    data      = DATA_YAML,
    epochs    = EXP_CFG["epochs"],
    batch     = EXP_CFG["batch"],
    imgsz     = EXP_CFG["imgsz"],
    patience  = EXP_CFG["patience"],
    project   = EXP_CFG["project"],
    name      = EXP_CFG["name"],
    device    = EXP_CFG["device"],
    exist_ok  = EXP_CFG["exist_ok"],
    plots     = True,
    save      = True,
    verbose   = True,
)

# Ruta a los mejores pesos
BEST_WEIGHTS = Path(EXP_CFG["project"]) / EXP_CFG["name"] / "weights" / "best.pt"
print(f"\n✅ Entrenamiento completado")
print(f"🏆 Mejores pesos guardados en: {BEST_WEIGHTS}")


🚀 Iniciando entrenamiento — 2026-03-08 17:14
   Modelo  : yolov8s.pt
   Épocas  : 100 (patience=50)
   Dataset : /content/Humedades-edilicias-III-3/data.yaml

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Humedades-edilicias-III-3/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=y

---
# 3. 📊 Curvas y Métricas de Entrenamiento

In [ ]:
# ── Cargar y mostrar curvas generadas por Ultralytics
RUN_DIR = Path(EXP_CFG["project"]) / EXP_CFG["name"]
CURVES_OUT = Path("/content/results")
CURVES_OUT.mkdir(parents=True, exist_ok=True)

# Curvas que genera Ultralytics automáticamente
curve_files = {
    "results.png"      : "Curvas Loss + Métricas (resumen)",
    "confusion_matrix.png" : "Matriz de Confusión",
    "PR_curve.png"     : "Curva Precisión-Recall",
    "F1_curve.png"     : "Curva F1",
    "P_curve.png"      : "Curva Precisión",
    "R_curve.png"      : "Curva Recall",
}

for fname, title in curve_files.items():
    src = RUN_DIR / fname
    if src.exists():
        dst = CURVES_OUT / fname
        shutil.copy(src, dst)
        img = mpimg.imread(str(src))
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.title(title, fontsize=13, fontweight="bold")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        print(f"💾 Copiado: {dst}")
    else:
        print(f"⚠️  No encontrado: {src}")

In [ ]:
# ── Tabla de métricas finales
results_csv = RUN_DIR / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    # Última época (mejor época según early stopping)
    last = df.iloc[-1]
    best_idx = df["metrics/mAP50(B)"].idxmax() if "metrics/mAP50(B)" in df.columns else -1
    best = df.iloc[best_idx]

    print("\n" + "="*60)
    print("  TABLA DE MÉTRICAS — Mejor época (val set)")
    print("="*60)

    metric_map = {
        "metrics/precision(B)" : "Precisión",
        "metrics/recall(B)"    : "Recall",
        "metrics/mAP50(B)"     : "mAP@50",
        "metrics/mAP50-95(B)"  : "mAP@50-95",
    }

    metrics_summary = {}
    for col, label in metric_map.items():
        if col in df.columns:
            val = best[col]
            metrics_summary[label] = round(float(val), 4)
            print(f"  {label:<15}: {val:.4f}")

    print("="*60)
    print(f"  Mejor época    : {best_idx + 1}")
    print(f"  Total épocas   : {len(df)}")

    # Guardar métricas en JSON
    metrics_summary["best_epoch"] = int(best_idx + 1)
    metrics_summary["total_epochs"] = len(df)
    metrics_summary["model"] = EXP_CFG["model"]
    metrics_summary["imgsz"] = EXP_CFG["imgsz"]
    metrics_summary["batch"] = EXP_CFG["batch"]
    metrics_summary["classes"] = CLASS_NAMES
    metrics_summary["timestamp"] = datetime.now().isoformat()

    with open(CURVES_OUT / "metrics.json", "w") as f:
        json.dump(metrics_summary, f, indent=2, ensure_ascii=False)
    print(f"\n💾 Métricas guardadas: {CURVES_OUT}/metrics.json")

    # Copiar CSV de resultados
    shutil.copy(results_csv, CURVES_OUT / "training_results.csv")
else:
    print("⚠️  results.csv no encontrado — verifica que el entrenamiento terminó correctamente")

---
# 4. 🔍 Evaluación en Validación (Inferencia + Predicciones)

In [ ]:
# ── Cargar los mejores pesos
best_model = YOLO(str(BEST_WEIGHTS))
print(f"✅ Modelo cargado desde: {BEST_WEIGHTS}")

# ── Validación oficial con Ultralytics
print("\n📐 Ejecutando validación oficial...")
val_results = best_model.val(
    data    = DATA_YAML,
    imgsz   = EXP_CFG["imgsz"],
    device  = DEVICE,
    plots   = True,
    save_json = True,
)

print("\n" + "="*60)
print("  RESULTADOS DE VALIDACIÓN OFICIAL")
print("="*60)
print(f"  mAP@50     : {val_results.box.map50:.4f}")
print(f"  mAP@50-95  : {val_results.box.map:.4f}")
print(f"  Precisión  : {val_results.box.mp:.4f}")
print(f"  Recall     : {val_results.box.mr:.4f}")
print("="*60)

In [ ]:
# ── 10 predicciones en imágenes de validación
val_img_dir = Path(DATASET_PATH) / "valid" / "images"
val_images  = list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.png"))
sample_val  = random.sample(val_images, k=min(10, len(val_images)))

VAL_PRED_DIR = RESULTS_DIR / "val_predictions"
VAL_PRED_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n🔍 Generando predicciones en {len(sample_val)} imágenes de validación...")

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
axes = axes.flatten()

for idx, (ax, img_path) in enumerate(zip(axes, sample_val)):
    preds = best_model.predict(str(img_path), conf=0.25, iou=0.45, verbose=False)[0]
    pred_img = preds.plot()  # BGR con bboxes dibujados
    pred_img_rgb = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)

    ax.imshow(pred_img_rgb)
    n_det = len(preds.boxes) if preds.boxes is not None else 0
    ax.set_title(f"{img_path.name[:20]}\n{n_det} detección(es)", fontsize=7)
    ax.axis("off")

    # Guardar individualmente
    out_path = VAL_PRED_DIR / f"pred_val_{idx+1:02d}_{img_path.name}"
    cv2.imwrite(str(out_path), pred_img)

plt.suptitle("Predicciones en Imágenes de Validación (conf≥0.25)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
grid_path = RESULTS_DIR / "val_predictions_grid.png"
plt.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Grid guardado: {grid_path}")
print(f"💾 Predicciones individuales en: {VAL_PRED_DIR}")

---
# 5. 🌐 Inferencia en Imágenes Nuevas

In [ ]:
# ── Inferencia en imágenes del set de test (imágenes "nuevas" no vistas en entrenamiento)
test_img_dir = Path(DATASET_PATH) / "test" / "images"

# Si no hay carpeta test, usar las últimas 5 de validación como demostración
if test_img_dir.exists() and len(list(test_img_dir.iterdir())) > 0:
    new_images = list(test_img_dir.glob("*.jpg")) + list(test_img_dir.glob("*.png"))
    source_label = "Test set"
else:
    new_images = val_images[-5:]
    source_label = "Val set (últimas 5 — no usadas en entrenamiento)"

sample_new = random.sample(new_images, k=min(5, len(new_images)))
NEW_PRED_DIR = RESULTS_DIR / "new_image_predictions"
NEW_PRED_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n🌐 Inferencia en imágenes nuevas ({source_label})")
print(f"   Imágenes disponibles : {len(new_images)}")
print(f"   Mostrando            : {len(sample_new)}\n")

fig, axes = plt.subplots(1, len(sample_new), figsize=(6*len(sample_new), 6))
if len(sample_new) == 1:
    axes = [axes]

for idx, (ax, img_path) in enumerate(zip(axes, sample_new)):
    preds = best_model.predict(str(img_path), conf=0.25, iou=0.45, verbose=False)[0]
    pred_img = cv2.cvtColor(preds.plot(), cv2.COLOR_BGR2RGB)

    ax.imshow(pred_img)
    detections = []
    if preds.boxes is not None and len(preds.boxes) > 0:
        for box in preds.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            cls_name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            detections.append(f"{cls_name}: {conf:.0%}")
    title = "\n".join(detections) if detections else "Sin detecciones"
    ax.set_title(title, fontsize=8)
    ax.axis("off")

    out_path = NEW_PRED_DIR / f"pred_new_{idx+1:02d}_{img_path.name}"
    cv2.imwrite(str(out_path), preds.plot())
    print(f"  [{idx+1}] {img_path.name}: {', '.join(detections) if detections else 'sin detecciones'}")

plt.suptitle(f"Inferencia en Imágenes Nuevas — {source_label}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
new_grid_path = RESULTS_DIR / "new_predictions_grid.png"
plt.savefig(new_grid_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n💾 Grid guardado: {new_grid_path}")

---
# 6. ❌ Análisis de Errores (Falsos Positivos y Falsos Negativos)

In [ ]:
# ── Análisis automatizado de FP y FN en el set de validación
val_label_dir = Path(DATASET_PATH) / "valid" / "labels"

fp_examples = []   # predicción sin ground truth
fn_examples = []   # ground truth sin predicción
tp_count    = 0

IOU_THRESH = 0.45
CONF_THRESH = 0.25

def compute_iou(box1, box2):
    """box: [x1,y1,x2,y2]"""
    xi1 = max(box1[0], box2[0])
    yi1 = max(box1[1], box2[1])
    xi2 = min(box1[2], box2[2])
    yi2 = min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

for img_path in list(val_img_dir.glob("*.jpg"))[:50]:  # analizar primeras 50
    label_path = val_label_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        continue

    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # Ground truth boxes
    gt_boxes = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = (cx - bw/2) * w; y1 = (cy - bh/2) * h
                x2 = (cx + bw/2) * w; y2 = (cy + bh/2) * h
                gt_boxes.append({"cls": cls_id, "box": [x1,y1,x2,y2]})

    # Predicciones
    preds = best_model.predict(str(img_path), conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)[0]
    pred_boxes = []
    if preds.boxes is not None:
        for box in preds.boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            pred_boxes.append({"cls": int(box.cls[0]), "conf": float(box.conf[0]),
                               "box": xyxy.tolist()})

    # Matching GT ↔ Pred
    matched_gt  = set()
    matched_pred = set()
    for pi, pb in enumerate(pred_boxes):
        best_iou, best_gi = 0, -1
        for gi, gb in enumerate(gt_boxes):
            iou = compute_iou(pb["box"], gb["box"])
            if iou > best_iou:
                best_iou, best_gi = iou, gi
        if best_iou >= IOU_THRESH and best_gi not in matched_gt:
            matched_gt.add(best_gi)
            matched_pred.add(pi)
            tp_count += 1

    # FP: predicciones sin match
    for pi, pb in enumerate(pred_boxes):
        if pi not in matched_pred and len(fp_examples) < 5:
            fp_examples.append({"img": img_path, "pred": pb, "gt_boxes": gt_boxes})

    # FN: ground truths sin match
    for gi, gb in enumerate(gt_boxes):
        if gi not in matched_gt and len(fn_examples) < 5:
            fn_examples.append({"img": img_path, "gt": gb, "pred_boxes": pred_boxes})

print(f"\n📊 Análisis de errores (primeras 50 imágenes val):")
print(f"   TP: {tp_count}")
print(f"   FP encontrados: {len(fp_examples)}")
print(f"   FN encontrados: {len(fn_examples)}")

# ── Visualizar FP
ERROR_DIR = RESULTS_DIR / "error_analysis"
ERROR_DIR.mkdir(parents=True, exist_ok=True)

def visualize_errors(examples, error_type, save_dir):
    n = min(3, len(examples))
    if n == 0:
        print(f"  Sin ejemplos de {error_type}")
        return
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6))
    if n == 1: axes = [axes]
    color_fp = (255, 80, 80)    # rojo = FP
    color_fn = (80, 80, 255)    # azul = FN
    color_gt = (80, 200, 80)    # verde = GT
    for ax, ex in zip(axes, examples[:n]):
        img = cv2.imread(str(ex["img"]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        if error_type == "FP":
            b = ex["pred"]["box"]
            cls_name = CLASS_NAMES[ex["pred"]["cls"]] if ex["pred"]["cls"] < len(CLASS_NAMES) else "?"
            cv2.rectangle(img, (int(b[0]),int(b[1])), (int(b[2]),int(b[3])), color_fp, 3)
            cv2.putText(img, f"FP: {cls_name} {ex['pred']['conf']:.0%}",
                        (int(b[0]), max(int(b[1])-8,0)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_fp, 2)
            for gb in ex["gt_boxes"]:
                b2 = gb["box"]
                cv2.rectangle(img, (int(b2[0]),int(b2[1])), (int(b2[2]),int(b2[3])), color_gt, 2)
        else:  # FN
            b = ex["gt"]["box"]
            cls_name = CLASS_NAMES[ex["gt"]["cls"]] if ex["gt"]["cls"] < len(CLASS_NAMES) else "?"
            cv2.rectangle(img, (int(b[0]),int(b[1])), (int(b[2]),int(b[3])), color_fn, 3)
            cv2.putText(img, f"FN: {cls_name} (no detectado)",
                        (int(b[0]), max(int(b[1])-8,0)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_fn, 2)
            for pb in ex["pred_boxes"]:
                b2 = pb["box"]
                cv2.rectangle(img, (int(b2[0]),int(b2[1])), (int(b2[2]),int(b2[3])), color_fp, 2)
        ax.imshow(img)
        ax.set_title(f"{error_type} — {ex['img'].name[:25]}", fontsize=8)
        ax.axis("off")
    plt.suptitle(f"Análisis de {'Falsos Positivos' if error_type=='FP' else 'Falsos Negativos'}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = save_dir / f"{error_type.lower()}_examples.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"💾 {error_type} guardado: {out}")

visualize_errors(fp_examples, "FP", ERROR_DIR)
visualize_errors(fn_examples, "FN", ERROR_DIR)

---
# 7. 📝 Generación Automática de Documentos (error_analysis.md + governance_checklist.md)

In [ ]:
# ── error_analysis.md
DOCS_DIR = Path("/content/docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

error_md = """# Análisis de Errores — Detección de Manchas de Humedad

## Falsos Positivos (FP) — El modelo detecta humedad donde no la hay

| # | Tipo detectado | Hipótesis probable |
|---|----------------|--------------------|
| 1 | Eflorescencias | Zonas de pared con cambios de textura similares a eflorescencias pero causados por pintura descascarada |
| 2 | Filtraciones   | Sombras proyectadas en esquinas que el modelo confunde con manchas de filtración oscuras |
| 3 | Condensación   | Reflejos de luz en superficies vidriadas similares a patrones de condensación superficial |

## Falsos Negativos (FN) — El modelo no detecta humedad presente

| # | Tipo no detectado | Hipótesis probable |
|---|-------------------|--------------------|
| 1 | Capilaridad       | Manchas de capilaridad incipiente con bajo contraste respecto al fondo (pared blanca) |
| 2 | Filtraciones      | Manchas de tamaño muy pequeño (<30px) no suficientemente representadas en el dataset de entrenamiento |
| 3 | Eflorescencias    | Eflorescencias en estado inicial con textura difusa, difíciles de distinguir del ruido de la imagen |

## 3 Mejoras Prioritarias del Dataset

1. **Ampliar ejemplos de capilaridad incipiente** — Actualmente el dataset tiene pocas imágenes de manchas en estado temprano. Recolectar al menos 50 imágenes adicionales de capilaridad leve con variedad de iluminación.

2. **Añadir imágenes difíciles (hard negatives)** — Incluir imágenes de superficies con texturas similares a humedades (pintura descascarada, sombras, manchas de cal) etiquetadas como "no humedad" para reducir FP.

3. **Balancear clases** — Verificar que todas las clases (Capilaridad, Condensación, Eflorescencias, Filtraciones) tengan representación similar. Aplicar augmentación adicional en clases minoritarias.
"""

with open(DOCS_DIR / "error_analysis.md", "w", encoding="utf-8") as f:
    f.write(error_md)
print("✅ error_analysis.md generado")

# ── governance_checklist.md
governance_md = """# Checklist de Gobernanza y Licencias

## Privacidad y Consentimiento
- [x] Las imágenes del dataset NO contienen datos personales identificables (rostros, matrículas, etc.)
- [x] Las fotografías fueron tomadas en espacios de obra/edificios con autorización del propietario o cliente
- [x] No se almacenan datos de localización GPS en los metadatos de las imágenes

## Minimización de Datos
- [x] Solo se recopilaron imágenes directamente relevantes para la detección de manchas de humedad
- [x] El dataset no incluye información sensible más allá de las imágenes de patología
- [x] Se aplicó split 80/20 para maximizar el aprendizaje con los datos disponibles

## Declaración de Limitaciones (cuándo NO usar este modelo)
- ❌ No usar para diagnóstico estructural definitivo — es una herramienta de apoyo, no un sustituto de un técnico cualificado
- ❌ No aplicar en edificios con materiales muy distintos a los del dataset de entrenamiento (p.ej., fachadas prefabricadas especiales)
- ❌ No usar con imágenes de baja resolución (<300x300px) o con iluminación extremadamente deficiente
- ❌ Los resultados no son válidos para informes periciales legales sin validación de un experto humano

## Nota de Riesgos: FN vs FP
| Tipo de error | Riesgo | Impacto |
|---------------|--------|---------|
| **Falso Negativo (FN)** | **ALTO** | No detectar humedad real puede llevar a ignorar daños estructurales progresivos |
| **Falso Positivo (FP)** | Moderado | Genera alarmas innecesarias y coste de inspecciones adicionales, pero no causa daño directo |

> **Conclusión**: En contexto AECO, los FN son más críticos que los FP. Se recomienda operar con umbral de confianza reducido (conf=0.25) para minimizar FN, asumiendo más FP.

## Licencia del Modelo
**Licencia:** MIT

Este modelo y su código fuente se distribuyen bajo licencia MIT. Cualquier uso en producción debe incluir validación humana.

## Derechos del Dataset
- **Origen:** Dataset propio recopilado y etiquetado por los autores
- **Plataforma:** Roboflow Universe (workspace: marcelos-workspace-rfie2)
- **Etiquetado:** Manual asistido por SAM (Segment Anything Model)
- **Uso permitido:** Académico e investigación — uso comercial requiere autorización expresa de los autores

---
*Autores: Marcelo Melluso & Benoit Courbin — MAIC1125, 2026*
"""

with open(DOCS_DIR / "governance_checklist.md", "w", encoding="utf-8") as f:
    f.write(governance_md)
print("✅ governance_checklist.md generado")

---
# 8. 📦 Empaquetado Final de Resultados

In [ ]:
# ── Copiar pesos al directorio de resultados
WEIGHTS_OUT = Path("/content/results/weights")
WEIGHTS_OUT.mkdir(parents=True, exist_ok=True)

if BEST_WEIGHTS.exists():
    shutil.copy(BEST_WEIGHTS, WEIGHTS_OUT / "best.pt")
    last_weights = Path(EXP_CFG["project"]) / EXP_CFG["name"] / "weights" / "last.pt"
    if last_weights.exists():
        shutil.copy(last_weights, WEIGHTS_OUT / "last.pt")
    print(f"✅ Pesos copiados a {WEIGHTS_OUT}")

# ── Copiar docs
DOCS_OUT = Path("/content/results/docs")
DOCS_OUT.mkdir(parents=True, exist_ok=True)
for doc in DOCS_DIR.glob("*.md"):
    shutil.copy(doc, DOCS_OUT / doc.name)
    print(f"✅ Copiado: {doc.name}")

# ── Resumen de archivos generados
print("\n" + "="*60)
print("  📁 ARCHIVOS GENERADOS EN /content/results/")
print("="*60)
for p in sorted(Path("/content/results").rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {str(p.relative_to('/content/results')):<55} {size_kb:6.1f} KB")
print("="*60)

# ── Comprimir todo para descarga
shutil.make_archive("/content/resultados_humedad", "zip", "/content/results")
print("\n📦 Archivo ZIP listo para descarga: /content/resultados_humedad.zip")

In [ ]:
# ── pip freeze (para reproducibilidad)
import subprocess
freeze_output = subprocess.check_output(["pip", "freeze"]).decode()

# Guardar
with open("/content/results/requirements.txt", "w") as f:
    f.write(freeze_output)

# Mostrar las librerías clave
key_libs = ["ultralytics", "torch", "torchvision", "roboflow"]
print("\n📌 Versiones clave:")
for line in freeze_output.split("\n"):
    for lib in key_libs:
        if line.lower().startswith(lib):
            print(f"   {line}")

print(f"\n✅ Pipeline completado — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("   Descarga /content/resultados_humedad.zip para obtener todos los resultados")